# 9-1-1 Dispatch Data: Time Series, Hypothesis Testing & Causal Inference

**Data**: `oz_test_incidents.csv` + `oz_test_hourly_call_counts.csv`
**Standards**: NENA-STA-020.1, APCO ANS 3.101, NFPA 1710

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as p9
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency, binomtest
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.diagnostic import acorr_ljungbox
import scikit_posthocs as sp

# Bayesian
import pymc as pm
import arviz as az
import bambi as bmb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
np.random.seed(42)
az.style.use('arviz-darkgrid')

In [ ]:
# Load Data
incidents = pd.read_csv("../oz_test_incidents.csv", 
    parse_dates=["call_start_time", "incident_start_time",
                 "time_phone_pickup", "time_call_enters_queue",
                 "time_first_unit_assigned", "time_unit_enroute",
                 "time_unit_arrived", "time_last_unit_cleared",
                 "time_call_closed", "time_phone_disconnect"])

calls = pd.read_csv("../oz_test_hourly_call_counts.csv", parse_dates=["hour_start"])

# Feature engineering
incidents["hour"] = incidents["call_start_time"].dt.hour
incidents["dow"] = incidents["call_start_time"].dt.dayofweek
incidents["is_weekend"] = incidents["dow"].isin([5, 6]).astype(int)
incidents["week"] = incidents["call_start_time"].dt.isocalendar().week
incidents["log_queue"] = np.log1p(incidents["dispatch_queue_seconds"])
incidents["log_total"] = np.log1p(incidents["total_elapsed_seconds"])

calls["hour"] = calls["hour_start"].dt.hour
calls["dow"] = calls["hour_start"].dt.dayofweek
calls["week"] = calls["hour_start"].dt.isocalendar().week

## 1. Time Series Diagnostics

In [ ]:
def ts_diagnostics(series, period=24, max_lag=48):
"""Complete time series diagnostic suite."""
    results = {}
    
    # Stationarity tests
    adf = adfuller(series.dropna())
    results["adf"] = {"stat": adf[0], "p": adf[1], "critical": adf[4]}
    
    kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct', nlags="auto")
    results["kpss"] = {"stat": kpss_stat, "p": kpss_p, "critical": kpss_crit}
    
    # ACF/PACF
    acf_vals = acf(series.dropna(), nlags=max_lag, fft=True)
    pacf_vals = pacf(series.dropna(), nlags=max_lag)
    results["acf"] = acf_vals
    results["pacf"] = pacf_vals
    
    # Ljung-Box test
    lb = acorr_ljungbox(series.dropna(), lags=[6, 12, 24, max_lag], return_df=True)
    results["ljung_box"] = lb
    
    # Seasonal decomposition
    stl = STL(series, period=period, robust=True)
    decomp = stl.fit()
    results["decomposition"] = {
        "trend": decomp.trend,
        "seasonal": decomp.seasonal,
        "resid": decomp.resid
    }
    
    # Strength
    var_resid = np.var(decomp.resid.dropna())
    var_trend_resid = np.var((decomp.trend + decomp.resid).dropna())
    var_seas_resid = np.var((decomp.seasonal + decomp.resid).dropna())
    
    results["trend_strength"] = max(0, 1 - var_resid / var_trend_resid)
    results["seasonal_strength"] = max(0, 1 - var_resid / var_seas_resid)
    
    return results

# Run on hourly 911 calls
ts_911 = calls.set_index("hour_start")["nine_one_one_calls_received"].asfreq("h")
diag_911 = ts_diagnostics(ts_911, period=24)

print("911 Time Series Diagnostics:")
print(f"  ADF: stat={diag_911['adf']['stat']:.3f}, p={diag_911['adf']['p']:.4f}")
print(f"  KPSS: stat={diag_911['kpss']['stat']:.3f}, p={diag_911['kpss']['p']:.4f}")
print(f"  Trend Strength: {diag_911['trend_strength']:.3f}")
print(f"  Seasonal Strength: {diag_911['seasonal_strength']:.3f}")
print(f"  Ljung-Box (lag 24): p={diag_911['ljung_box'].iloc[2]['lb_pvalue']:.4f}")

In [ ]:
# Plot ACF/PACF
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_acf(ts_911.dropna(), lags=48, ax=axes[0,0], title="ACF - 911 Calls")
plot_pacf(ts_911.dropna(), lags=48, ax=axes[0,1], title="PACF - 911 Calls")
plot_acf(calls.set_index("hour_start")["non_emergency_calls_received"].dropna(), lags=48, ax=axes[1,0], title="ACF - Non-Emergency")
plot_pacf(calls.set_index("hour_start")["non_emergency_calls_received"].dropna(), lags=48, ax=axes[1,1], title="PACF - Non-Emergency")
plt.tight_layout()
plt.show()

In [ ]:
# STL Decomposition
stl = STL(ts_911, period=24, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(14, 10)
plt.suptitle("STL Decomposition: 911 Call Volume (Period=24h)")
plt.tight_layout()
plt.show()

## 2. Forecasting Model Suite

In [ ]:
def forecasting_suite(train, test=None, period=24, horizon=24):
"""Fit and compare multiple forecasting models."""
    models = {}
    forecasts = {}
    metrics = {}
    
    # 1. Naive
    naive_fc = np.full(horizon, train.iloc[-1])
    models["Naive"] = "Last value"
    forecasts["Naive"] = naive_fc
    
    # 2. Seasonal Naive
    seas_naive = train.iloc[-period*7:].values[-horizon:] if len(train) >= period*7 else naive_fc
    models["Seasonal_Naive"] = "Same hour last week"
    forecasts["Seasonal_Naive"] = seas_naive
    
    # 3. Holt-Winters
    try:
        hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=period).fit()
        models["Holt_Winters"] = hw
        forecasts["Holt_Winters"] = hw.forecast(horizon)
    except Exception as e:
        print(f"Holt-Winters failed: {e}")
    
    # 4. ARIMA (auto)
    try:
        best_aic = np.inf
        best_order = None
        for p in range(3):
            for d in range(2):
                for q in range(3):
                    try:
                        m = ARIMA(train, order=(p,d,q)).fit()
                        if m.aic < best_aic:
                            best_aic = m.aic
                            best_order = (p,d,q)
                    except:
                        pass
        if best_order:
            arima = ARIMA(train, order=best_order).fit()
            models["ARIMA"] = arima
            forecasts["ARIMA"] = arima.forecast(horizon)
            print(f"Best ARIMA: {best_order}, AIC={best_aic:.1f}")
    except Exception as e:
        print(f"ARIMA failed: {e}")
    
    # Evaluate on test set
    if test is not None:
        for name, fc in forecasts.items():
            if len(fc) >= len(test):
                mae = np.mean(np.abs(fc[:len(test)] - test.values))
                rmse = np.sqrt(np.mean((fc[:len(test)] - test.values)**2))
                mape = np.mean(np.abs((fc[:len(test)] - test.values) / test.values)) * 100
                metrics[name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape}
    
    return models, forecasts, metrics

# Split: first 5 days train, last 2 days test
train_len = 5 * 24
train = ts_911[:train_len]
test = ts_911[train_len:]

models, forecasts, metrics = forecasting_suite(train, test, period=24, horizon=48)

print("Forecast Metrics (on test set):")
for name, m in metrics.items():
    print(f"  {name}: MAE={m['MAE']:.2f}, RMSE={m['RMSE']:.2f}, MAPE={m['MAPE']:.1f}%")

In [ ]:
# Plot forecasts
plt.figure(figsize=(14, 6))
plt.plot(test.index, test.values, 'k-', label='Actual', linewidth=2)
for name, fc in forecasts.items():
    if len(fc) >= len(test):
        plt.plot(test.index, fc[:len(test)], '--', label=name, alpha=0.7)
plt.legend()
plt.title("911 Call Volume Forecasts (Last 2 Days)")
plt.show()

## 3. Bayesian Structural Time Series (BSTS)

In [ ]:
def bayesian_structural_time_series(series, period=24, horizon=24):
"""
Bayesian Structural Time Series with local linear trend and seasonal.
"""
    y = series.values
    y_std = (y - y.mean()) / y.std()
    n = len(y_std)
    
    with pm.Model() as model:
        # Observation noise
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.1)
        
        # Local linear trend
        sigma_level = pm.HalfNormal("sigma_level", sigma=0.01)
        level = pm.GaussianRandomWalk("level", sigma=sigma_level, init_dist=pm.Normal.dist(0, 1), shape=n)
        
        sigma_slope = pm.HalfNormal("sigma_slope", sigma=0.001)
        slope = pm.GaussianRandomWalk("slope", sigma=sigma_slope, init_dist=pm.Normal.dist(0, 0.1), shape=n)
        
        trend = pm.Deterministic("trend", level + pm.math.cumsum(slope))
        
        # Seasonal (Fourier terms)
        n_harmonics = min(6, period // 2)
        t = np.arange(n)
        seasonal = 0
        for k in range(1, n_harmonics + 1):
            a_k = pm.Normal(f"a_{k}", mu=0, sigma=0.1)
            b_k = pm.Normal(f"b_{k}", mu=0, sigma=0.1)
            seasonal += a_k * np.sin(2 * np.pi * k * t / period) + b_k * np.cos(2 * np.pi * k * t / period)
        
        seasonal = pm.Deterministic("seasonal", seasonal)
        
        mu = pm.Deterministic("mu", trend + seasonal)
        
        likelihood = pm.Normal("y", mu=mu, sigma=sigma_obs, observed=y_std)
        
        trace = pm.sample(2000, tune=1500, chains=4, target_accept=0.95, random_seed=42)
    
    # Forecast
    with model:
        post_level = trace.posterior["level"][:, :, -1].values
        post_slope = trace.posterior["slope"][:, :, -1].values
        post_trend = trace.posterior["trend"][:, :, -1].values
        
        # Seasonal forecast
        t_future = np.arange(n, n + horizon)
        seasonal_future = 0
        for k in range(1, n_harmonics + 1):
            a_k = trace.posterior[f"a_{k}"].mean(dim=["chain", "draw"]).values
            b_k = trace.posterior[f"b_{k}"].mean(dim=["chain", "draw"]).values
            seasonal_future += a_k * np.sin(2 * np.pi * k * t_future / period) + \
                              b_k * np.cos(2 * np.pi * k * t_future / period)
        
        # Trend forecast
        trend_future = post_trend[:, None] + np.cumsum(post_slope[:, None] * np.ones(horizon), axis=1)
        
        mu_future = trend_future + seasonal_future
        
        sigma_obs_post = trace.posterior["sigma_obs"].mean().values
        forecast_samples = np.random.normal(mu_future, sigma_obs_post)
    
    # Rescale
    forecast_mean = forecast_samples.mean(axis=0) * y.std() + y.mean()
    forecast_lower = np.percentile(forecast_samples, 2.5, axis=0) * y.std() + y.mean()
    forecast_upper = np.percentile(forecast_samples, 97.5, axis=0) * y.std() + y.mean()
    
    return trace, model, forecast_mean, forecast_lower, forecast_upper

# Run BSTS on last 3 days
train_bsts = ts_911[-72:]
trace_bsts, model_bsts, fc_mean, fc_lower, fc_upper = bayesian_structural_time_series(train_bsts, period=24, horizon=24)

In [ ]:
# Plot BSTS forecast
plt.figure(figsize=(12, 5))
hist_idx = train_bsts.index
future_idx = pd.date_range(hist_idx[-1] + pd.Timedelta(hours=1), periods=24, freq="h")
plt.plot(hist_idx, train_bsts.values, 'k-', label='History', alpha=0.5)
plt.plot(future_idx, fc_mean, 'b-', label='Forecast')
plt.fill_between(future_idx, fc_lower, fc_upper, alpha=0.2, color='blue', label='95% CI')
plt.title("Bayesian Structural Time Series Forecast (911 Calls)")
plt.legend()
plt.show()

## 4. Compliance Hypothesis Testing

In [ ]:
def compliance_hypothesis_tests(incidents, calls):
"""Comprehensive compliance hypothesis tests against standards."""
    results = {}
    
    # NENA 9-1-1 Answer Time
    total_911 = calls["nine_one_one_calls_received"].sum()
    ans_15 = int((calls["nine_one_one_answered_15s_pct"]/100 * calls["nine_one_one_calls_received"]).sum())
    ans_20 = int((calls["nine_one_one_answered_20s_pct"]/100 * calls["nine_one_one_calls_received"]).sum())
    ans_10 = int((calls["nine_one_one_answered_10s_pct"]/100 * calls["nine_one_one_calls_received"]).sum())
    
    test_15 = binomtest(ans_15, total_911, p=0.90, alternative="greater")
    test_20 = binomtest(ans_20, total_911, p=0.95, alternative="greater")
    test_10 = binomtest(ans_10, total_911, p=0.75, alternative="greater")
    
    ci_15 = proportion_confint(ans_15, total_911, method="wilson")
    ci_20 = proportion_confint(ans_20, total_911, method="wilson")
    ci_10 = proportion_confint(ans_10, total_911, method="wilson")
    
    results["NENA_15s"] = {"observed": ans_15/total_911, "target": 0.90, 
                           "p_value": test_15.pvalue, "ci": ci_15}
    results["NENA_20s"] = {"observed": ans_20/total_911, "target": 0.95, 
                           "p_value": test_20.pvalue, "ci": ci_20}
    results["APCO_10s"] = {"observed": ans_10/total_911, "target": 0.75, 
                           "p_value": test_10.pvalue, "ci": ci_10}
    
    # NFPA 1710 Dispatch Queue
    dq = incidents["dispatch_queue_seconds"].dropna()
    pct_64 = (dq <= 64).mean()
    pct_106 = (dq <= 106).mean()
    
    test_64 = binomtest(int(pct_64*len(dq)), len(dq), p=0.90, alternative="greater")
    test_106 = binomtest(int(pct_106*len(dq)), len(dq), p=0.95, alternative="greater")
    
    ci_64 = proportion_confint(int(pct_64*len(dq)), len(dq), method="wilson")
    ci_106 = proportion_confint(int(pct_106*len(dq)), len(dq), method="wilson")
    
    results["NFPA_DQ_64"] = {"observed": pct_64, "target": 0.90, 
                             "p_value": test_64.pvalue, "ci": ci_64}
    results["NFPA_DQ_106"] = {"observed": pct_106, "target": 0.95, 
                              "p_value": test_106.pvalue, "ci": ci_106}
    
    # NFPA 1710 Turnout
    for agency, target in [("EMS", 60), ("FIRE", 80)]:
        turnout = incidents[incidents["agency"]==agency]["turnout_seconds"].dropna()
        pct = (turnout <= target).mean()
        test = binomtest(int(pct*len(turnout)), len(turnout), p=0.90, alternative="greater")
        ci = proportion_confint(int(pct*len(turnout)), len(turnout), method="wilson")
        results[f"NFPA_Turnout_{agency}"] = {"observed": pct, "target": 0.90, 
                                             "p_value": test.pvalue, "ci": ci}
    
    # NFPA 1710 Travel
    travel = incidents["travel_seconds"].dropna()
    pct_240 = (travel <= 240).mean()
    test_240 = binomtest(int(pct_240*len(travel)), len(travel), p=0.90, alternative="greater")
    ci_240 = proportion_confint(int(pct_240*len(travel)), len(travel), method="wilson")
    results["NFPA_Travel_240"] = {"observed": pct_240, "target": 0.90, 
                                  "p_value": test_240.pvalue, "ci": ci_240}
    
    return results

comp_results = compliance_hypothesis_tests(incidents, calls)

print("COMPLIANCE HYPOTHESIS TESTS (H₀: rate ≤ target)")
print("=" * 80)
for test, res in comp_results.items():
    status = "PASS" if res["p_value"] < 0.05 else "FAIL"
    print(f"{test}:")
    print(f"  Observed: {res['observed']:.3f}, Target: {res['target']:.2f}")
    print(f"  p-value: {res['p_value']:.6f} → {status}")
    print(f"  95% CI: [{res['ci'][0]:.3f}, {res['ci'][1]:.3f}]")
    print()

## 5. Group Comparisons with Effect Sizes

In [ ]:
def group_comparison_suite(data, outcome_col, group_col, parametric=False):
"""Comprehensive group comparison with effect sizes."""
    groups = data[group_col].unique()
    group_data = [data[data[group_col]==g][outcome_col].dropna() for g in groups]
    
    results = {"groups": groups, "n_per_group": [len(g) for g in group_data]}
    
    if parametric:
        f_stat, p_val = stats.f_oneway(*group_data)
        results["anova"] = {"F": f_stat, "p": p_val}
        from statsmodels.stats.multicomp import pairwise_tukeyhsd
        tukey = pairwise_tukeyhsd(data[outcome_col].dropna(), data[group_col].dropna())
        results["tukey"] = tukey
    else:
        h_stat, p_val = stats.kruskal(*group_data)
        results["kruskal"] = {"H": h_stat, "p": p_val}
        
        dunn = sp.posthoc_dunn(data, val_col=outcome_col, group_col=group_col, p_adjust="bonferroni")
        results["dunn"] = dunn
    
    # Effect sizes
    if parametric:
        ss_between = sum(len(g) * (g.mean() - data[outcome_col].mean())**2 for g in group_data)
        ss_total = sum((data[outcome_col] - data[outcome_col].mean())**2)
        eta_sq = ss_between / ss_total
        results["eta_squared"] = eta_sq
    else:
        n_total = sum(len(g) for g in group_data)
        epsilon_sq = (h_stat - len(groups) + 1) / (n_total - len(groups))
        results["epsilon_squared"] = max(0, epsilon_sq)
    
    # Pairwise Cliff's delta
    from itertools import combinations
    pairwise_effects = {}
    for (i, g1), (j, g2) in combinations(enumerate(groups), 2):
        cliff = pg.compute_effsize(group_data[i], group_data[j], eftype="cliff")
        pairwise_effects[f"{g1}_vs_{g2}"] = cliff
    results["pairwise_cliff"] = pairwise_effects
    
    return results

# Compare shifts on total elapsed time
shift_comp = group_comparison_suite(incidents, "total_elapsed_seconds", "shift_label", parametric=False)
print(f"Kruskal-Wallis: H={shift_comp['kruskal']['H']:.2f}, p={shift_comp['kruskal']['p']:.6f}")
print(f"Epsilon-squared: {shift_comp['epsilon_squared']:.4f}")
print("\nPairwise Cliff's Delta:")
for k, v in shift_comp['pairwise_cliff'].items():
    print(f"  {k}: {v:.3f}")

# Dunn post-hoc
print("\nDunn Post-hoc (Bonferroni):")
print(shift_comp['dunn'].round(4))

In [ ]:
# Trend Tests (Mann-Kendall)
def trend_tests(series, alpha=0.05):
"""Multiple trend tests for time series."""
    results = {}
    
    # Mann-Kendall
    n = len(series)
    s = 0
    for i in range(n-1):
        for j in range(i+1, n):
            s += np.sign(series[j] - series[i])
    
    var_s = n*(n-1)*(2*n+5)/18
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
    p_mk = 2 * (1 - stats.norm.cdf(abs(z)))
    
    # Sen's slope
    slopes = []
    for i in range(n-1):
        for j in range(i+1, n):
            slopes.append((series[j] - series[i]) / (j - i))
    sen_slope = np.median(slopes)
    
    results["mann_kendall"] = {"S": s, "Z": z, "p": p_mk, "sen_slope": sen_slope}
    
    # Linear regression trend
    x = np.arange(n)
    slope, intercept, r, p_lr, se = stats.linregress(x, series)
    results["linear_trend"] = {"slope": slope, "r": r, "p": p_lr, "se": se}
    
    return results

# Weekly compliance trend
weekly_compliance = calls.groupby("week")["nine_one_one_answered_20s_pct"].mean()
trend_results = trend_tests(weekly_compliance.values)

print("Weekly 20s Compliance Trend:")
for test, res in trend_results.items():
    print(f"  {test}: {res}")

## 6. Bayesian Compliance Testing (ROPE)

In [ ]:
def bayesian_compliance_test(successes, trials, target_rate=0.90, 
                             prior_alpha=90, prior_beta=10, 
                             rope_width=0.02):
"""
Bayesian compliance test with ROPE (Region of Practical Equivalence).
"""
    with pm.Model() as model:
        p = pm.Beta("p", alpha=prior_alpha, beta=prior_beta)
        observed = pm.Binomial("obs", n=trials, p=p, observed=successes)
        trace = pm.sample(3000, tune=1500, chains=4, target_accept=0.95, random_seed=42)
    
    post_p = trace.posterior["p"].values.flatten()
    
    prob_above = (post_p > target_rate).mean()
    
    rope_low = target_rate - rope_width
    rope_high = target_rate + rope_width
    in_rope = ((post_p >= rope_low) & (post_p <= rope_high)).mean()
    below_rope = (post_p < rope_low).mean()
    above_rope = (post_p > rope_high).mean()
    
    if prob_above > 0.95:
        decision = "Compliant (high confidence)"
    elif below_rope > 0.95:
        decision = "Non-compliant (high confidence)"
    else:
        decision = "Uncertain - more data needed"
    
    return {
        "trace": trace,
        "posterior_mean": post_p.mean(),
        "hdi_95": az.hdi(post_p, hdi_prob=0.95),
        "prob_above_target": prob_above,
        "rope": {"low": rope_low, "high": rope_high, 
                 "in_rope": in_rope, "below": below_rope, "above": above_rope},
        "decision": decision
    }

# Test NENA 15s compliance
total_911 = calls["nine_one_one_calls_received"].sum()
ans_15 = int((calls["nine_one_one_answered_15s_pct"]/100 * calls["nine_one_one_calls_received"]).sum())

bayes_comp = bayesian_compliance_test(ans_15, total_911, target_rate=0.90)
print("Bayesian NENA 15s Compliance:")
for k, v in bayes_comp.items():
    if k != "trace":
        print(f"  {k}: {v}")

In [ ]:
# Plot posterior with ROPE
az.plot_posterior(bayes_comp["trace"], var_names=["p"], 
                  rope=[0.88, 0.92], ref_val=0.90,
                  hdi_prob=0.95)
plt.suptitle("Bayesian NENA 15s Compliance (ROPE: 88%-92%)")
plt.show()

## 7. Bayesian Group Comparison (BEST)

In [ ]:
def bayesian_group_comparison(group1, group2, group_names=["Group 1", "Group 2"]):
"""
Bayesian estimation of difference between groups (BEST procedure).
Uses robust t-distribution for heavy tails.
"""
    y1 = group1.dropna().values
    y2 = group2.dropna().values
    
    with pm.Model() as model:
        mu1 = pm.Normal("mu1", mu=y1.mean(), sigma=y1.std()*2)
        mu2 = pm.Normal("mu2", mu=y2.mean(), sigma=y2.std()*2)
        sigma1 = pm.HalfNormal("sigma1", sigma=y1.std()*2)
        sigma2 = pm.HalfNormal("sigma2", sigma=y2.std()*2)
        nu = pm.Exponential("nu", lam=1/29) + 1
        
        like1 = pm.StudentT("like1", nu=nu, mu=mu1, sigma=sigma1, observed=y1)
        like2 = pm.StudentT("like2", nu=nu, mu=mu2, sigma=sigma2, observed=y2)
        
        diff_mu = pm.Deterministic("diff_mu", mu1 - mu2)
        diff_sigma = pm.Deterministic("diff_sigma", sigma1 - sigma2)
        effect_size = pm.Deterministic("effect_size", (mu1 - mu2) / pm.math.sqrt((sigma1**2 + sigma2**2)/2))
        
        trace = pm.sample(3000, tune=1500, chains=4, target_accept=0.95, random_seed=42)
    
    post_diff = trace.posterior["diff_mu"].values.flatten()
    post_eff = trace.posterior["effect_size"].values.flatten()
    
    return {
        "trace": trace,
        "diff_mean": post_diff.mean(),
        "diff_hdi": az.hdi(post_diff, hdi_prob=0.95),
        "effect_size_mean": post_eff.mean(),
        "effect_size_hdi": az.hdi(post_eff, hdi_prob=0.95),
        "prob_diff_positive": (post_diff > 0).mean(),
        "prob_eff_gt_02": (post_eff > 0.2).mean(),
        "prob_eff_gt_05": (post_eff > 0.5).mean(),
        "prob_eff_gt_08": (post_eff > 0.8).mean(),
    }

# Compare Day vs Night shift total elapsed time
day = incidents[incidents["shift"]=="D"]["total_elapsed_seconds"]
night = incidents[incidents["shift"]=="N"]["total_elapsed_seconds"]

bayes_grp = bayesian_group_comparison(day, night, ["Day", "Night"])
print("Bayesian Day vs Night Comparison:")
for k, v in bayes_grp.items():
    if k != "trace":
        print(f"  {k}: {v}")

In [ ]:
# Plot
az.plot_posterior(bayes_grp["trace"], var_names=["diff_mu", "effect_size"], hdi_prob=0.95)
plt.suptitle("Bayesian Group Comparison: Day vs Night Shift")
plt.show()

## 8. Causal Inference: Difference-in-Differences

In [ ]:
def did_analysis(data, treatment_col, time_col, outcome_col, group_col=None):
"""Difference-in-Differences with parallel trends check."""
    df = data.copy()
    df["post"] = (df[time_col] >= df[time_col].median()).astype(int)
    df["treated"] = df[treatment_col].astype(int)
    df["did"] = df["post"] * df["treated"]
    
    formula = f"{outcome_col} ~ post + treated + did"
    if group_col:
        formula += f" + C({group_col})"
    
    model = smf.ols(formula, data=df).fit()
    
    # Parallel trends test
    pre_data = df[df["post"] == 0]
    if len(pre_data) > 0:
        parallel_formula = f"{outcome_col} ~ treated * C({time_col})"
        if group_col:
            parallel_formula += f" + C({group_col})"
        parallel_test = smf.ols(parallel_formula, data=pre_data).fit()
        parallel_p = parallel_test.pvalues.filter(like="treated:").min() if any("treated:" in x for x in parallel_test.pvalues.index) else 1.0
    else:
        parallel_p = np.nan
    
    return {
        "model": model,
        "did_estimate": model.params.get("did", np.nan),
        "did_pvalue": model.pvalues.get("did", np.nan),
        "did_ci": model.conf_int().loc["did"] if "did" in model.conf_int().index else [np.nan, np.nan],
        "parallel_trends_p": parallel_p
    }

# Simulate: New dispatch protocol for LAW agency starting mid-week
incidents["week_day"] = incidents["call_start_time"].dt.dayofweek
incidents["post_protocol"] = (incidents["week_day"] >= 3).astype(int)  # Thu-Sun
incidents["treated_law"] = (incidents["agency"] == "LAW").astype(int)

did_result = did_analysis(
    incidents, "treated_law", "post_protocol", "log_queue", "priority"
)
print("DiD Results:")
for k, v in did_result.items():
    if k != "model":
        print(f"  {k}: {v}")

In [ ]:
# Interrupted Time Series
def its_analysis(data, time_col, outcome_col, intervention_time):
"""Interrupted Time Series analysis."""
    df = data.copy()
    df = df.sort_values(time_col).reset_index(drop=True)
    df["time_idx"] = range(len(df))
    df["post"] = (df[time_col] >= intervention_time).astype(int)
    df["time_post"] = df["time_idx"] * df["post"]
    
    formula = f"{outcome_col} ~ time_idx + post + time_post"
    model = smf.ols(formula, data=df).fit()
    
    return {
        "model": model,
        "level_change": model.params.get("post", np.nan),
        "slope_change": model.params.get("time_post", np.nan),
        "level_p": model.pvalues.get("post", np.nan),
        "slope_p": model.pvalues.get("time_post", np.nan)
    }

# Daily compliance ITS
daily_compliance = incidents.set_index("call_start_time").resample("D").agg(
    median_queue=("dispatch_queue_seconds", "median"),
    pct_queue_64=("dispatch_queue_seconds", lambda x: (x <= 64).mean()*100)
).reset_index()

intervention_time = daily_compliance["call_start_time"].iloc[3]
its_result = its_analysis(daily_compliance, "call_start_time", "median_queue", intervention_time)
print("ITS Results:")
for k, v in its_result.items():
    if k != "model":
        print(f"  {k}: {v}")

In [ ]:
# Propensity Score Matching
def propensity_score_matching(data, treatment_col, outcome_col, covariates, 
                              caliper=0.2, ratio=1):
"""Propensity score matching for causal effect estimation."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors
    
    df = data.dropna(subset=[treatment_col, outcome_col] + covariates).copy()
    
    X = pd.get_dummies(df[covariates], drop_first=True)
    y = df[treatment_col]
    
    ps_model = LogisticRegression(random_state=42, max_iter=1000)
    ps_model.fit(X, y)
    df["propensity"] = ps_model.predict_proba(X)[:, 1]
    
    treated = df[df[treatment_col] == 1]
    control = df[df[treatment_col] == 0]
    
    nn = NearestNeighbors(n_neighbors=ratio, metric="euclidean")
    nn.fit(control[["propensity"]].values)
    
    distances, indices = nn.kneighbors(treated[["propensity"]].values)
    
    matched_treated = []
    matched_control = []
    
    for i, (dist, idx) in enumerate(zip(distances, indices)):
        if dist[0] <= caliper * treated["propensity"].std():
            matched_treated.append(treated.iloc[i])
            matched_control.append(control.iloc[idx[0]])
    
    matched_treated = pd.DataFrame(matched_treated)
    matched_control = pd.DataFrame(matched_control)
    
    # Balance check
    balance = {}
    for cov in covariates:
        if cov in matched_treated.columns:
            std_diff = (matched_treated[cov].mean() - matched_control[cov].mean()) / \
                       np.sqrt((matched_treated[cov].var() + matched_control[cov].var()) / 2)
            balance[cov] = std_diff
    
    att = matched_treated[outcome_col].mean() - matched_control[outcome_col].mean()
    
    # Bootstrap CI
    boot_atts = []
    for _ in range(1000):
        bt = matched_treated.sample(len(matched_treated), replace=True)
        bc = matched_control.sample(len(matched_control), replace=True)
        boot_atts.append(bt[outcome_col].mean() - bc[outcome_col].mean())
    
    ci = np.percentile(boot_atts, [2.5, 97.5])
    
    return {
        "matched_treated": matched_treated,
        "matched_control": matched_control,
        "balance": balance,
        "att": att,
        "att_ci": ci,
        "n_matched": len(matched_treated),
    }

# Effect of Priority 1 vs 2 on dispatch queue
priority_data = incidents[incidents["priority"].isin([1, 2])].copy()
priority_data["treatment"] = (priority_data["priority"] == 1).astype(int)

covariates = ["agency", "problem_nature", "hour", "shift_label", "dow"]
covariates = [c for c in covariates if c in priority_data.columns]

psm_result = propensity_score_matching(
    priority_data, "treatment", "dispatch_queue_seconds", covariates
)

print(f"PSM ATT: {psm_result['att']:.1f}s (95% CI: {psm_result['att_ci'][0]:.1f}, {psm_result['att_ci'][1]:.1f})")
print(f"Matched pairs: {psm_result['n_matched']}")
print("Balance (std diff < 0.1 is good):")
for cov, diff in psm_result["balance"].items():
    print(f"  {cov}: {diff:.3f}")

In [ ]:
# Regression Discontinuity Design
def rdd_analysis(data, running_var, outcome_col, cutoff, bandwidth=None):
"""Sharp Regression Discontinuity Design."""
    df = data.dropna(subset=[running_var, outcome_col]).copy()
    df["above"] = (df[running_var] >= cutoff).astype(int)
    df["distance"] = df[running_var] - cutoff
    
    if bandwidth is None:
        bandwidth = 1.5 * df[running_var].std()
    
    local = df[np.abs(df["distance"]) <= bandwidth].copy()
    local["above_dist"] = local["above"] * local["distance"]
    
    model = smf.ols(f"{outcome_col} ~ above + distance + above_dist", data=local).fit()
    
    # Robustness across bandwidths
    bandwidths = [bandwidth * 0.5, bandwidth, bandwidth * 1.5, bandwidth * 2]
    robustness = {}
    for bw in bandwidths:
        local_bw = df[np.abs(df["distance"]) <= bw]
        if len(local_bw) > 20:
            m = smf.ols(f"{outcome_col} ~ above + distance + above_dist", data=local_bw).fit()
            robustness[bw] = {"estimate": m.params.get("above", np.nan), 
                              "p": m.pvalues.get("above", np.nan),
                              "n": len(local_bw)}
    
    # Placebo test
    fake_cutoff = cutoff + bandwidth
    df["fake_above"] = (df[running_var] >= fake_cutoff).astype(int)
    df["fake_distance"] = df[running_var] - fake_cutoff
    fake_local = df[np.abs(df["fake_distance"]) <= bandwidth]
    placebo = np.nan
    if len(fake_local) > 20:
        fake_model = smf.ols(f"{outcome_col} ~ fake_above + fake_distance + fake_above:fake_distance", 
                            data=fake_local).fit()
        placebo = fake_model.params.get("fake_above", np.nan)
    
    return {
        "model": model,
        "estimate": model.params.get("above", np.nan),
        "p_value": model.pvalues.get("above", np.nan),
        "ci": model.conf_int().loc["above"] if "above" in model.conf_int().index else [np.nan, np.nan],
        "bandwidth": bandwidth,
        "n_local": len(local),
        "robustness": robustness,
        "placebo": placebo
    }

# Compliance at queue threshold
calls_rdd = calls.copy()
calls_rdd["queue_proxy"] = calls_rdd["nine_one_one_calls_received"]
calls_rdd["outcome"] = calls_rdd["nine_one_one_answered_20s_pct"]

rdd_result = rdd_analysis(calls_rdd, "queue_proxy", "outcome", cutoff=5)
print("RDD Results (Compliance at queue=5):")
for k, v in rdd_result.items():
    if k not in ["model", "robustness"]:
        print(f"  {k}: {v}")
print("\nRobustness across bandwidths:")
for bw, res in rdd_result["robustness"].items():
    print(f"  BW={bw:.1f}: est={res['estimate']:.2f}, p={res['p']:.4f}, n={res['n']}")

## 9. Bayesian DiD

In [ ]:
def bayesian_did(data, treatment_col, time_col, outcome_col, group_col=None):
"""Bayesian Difference-in-Differences."""
    df = data.copy()
    df["post"] = (df[time_col] >= df[time_col].median()).astype(int)
    df["treated"] = df[treatment_col].astype(int)
    df["did"] = df["post"] * df["treated"]
    
    y = df[outcome_col].values
    if stats.skew(y) > 1:
        y = np.log1p(y)
    
    with pm.Model() as model:
        intercept = pm.Normal("intercept", mu=y.mean(), sigma=y.std()*2)
        beta_post = pm.Normal("beta_post", mu=0, sigma=1)
        beta_treated = pm.Normal("beta_treated", mu=0, sigma=1)
        beta_did = pm.Normal("beta_did", mu=0, sigma=1)
        
        if group_col:
            groups = df[group_col].unique()
            n_groups = len(groups)
            group_idx = pd.Categorical(df[group_col]).codes
            beta_group = pm.Normal("beta_group", mu=0, sigma=1, shape=n_groups)
            group_effect = beta_group[group_idx]
        else:
            group_effect = 0
        
        mu = (intercept + 
              beta_post * df["post"] + 
              beta_treated * df["treated"] + 
              beta_did * df["did"] + 
              group_effect)
        
        sigma = pm.HalfNormal("sigma", sigma=y.std())
        likelihood = pm.Normal("y", mu=mu, sigma=sigma, observed=y)
        
        trace = pm.sample(2000, tune=1500, chains=4, target_accept=0.95, random_seed=42)
    
    post_did = trace.posterior["beta_did"].values.flatten()
    
    return {
        "trace": trace,
        "did_mean": post_did.mean(),
        "did_hdi": az.hdi(post_did, hdi_prob=0.95),
        "prob_positive": (post_did > 0).mean(),
        "prob_neg": (post_did < 0).mean()
    }

# bayes_did = bayesian_did(incidents, "treated_law", "post_protocol", "dispatch_queue_seconds", "priority")
# print(f"Bayesian DiD: {bayes_did['did_mean']:.3f} (95% HDI: {bayes_did['did_hdi']})")
# print(f"P(DiD > 0): {bayes_did['prob_positive']:.3f}")